In [1]:
import os

from src.models import SearchCriteria
from src.setup import CONFIG_DIR
from src.utils import load_yaml

In [2]:
criteria_filepath = os.path.join(CONFIG_DIR, "criteria.yaml")
criteria = load_yaml(criteria_filepath, SearchCriteria).search_parameters

params = criteria[0]


In [ ]:
from src.models import LLMUserProfileModel
from src.parser import match_jobs
from src.scraper import find_new_jobs
from src.setup import DATA_DIR

file_exists = False
processed_ids = set()

profile_filepath = os.path.join(CONFIG_DIR, 'profile.yaml')
profile = load_yaml(profile_filepath, LLMUserProfileModel)

data_filepath = os.path.join(DATA_DIR, "jobs.csv")
new_jobs, ids = find_new_jobs(params)
jobs_filtered = [job for job in new_jobs
                    if job.job_id not in processed_ids]
new_jobs_df = match_jobs(profile, jobs_filtered)
processed_ids |= ids
new_jobs_df.to_csv(data_filepath,
                    mode='a',
                    header=not file_exists,
                    index=False)
file_exists = True
file_exists = True

2026-05-20 21:33:13,276 [INFO] src.parser - Loading local LLM: llama3.2:3b
Matching jobs: 0it [00:00, ?it/s]


In [11]:
import pandas as pd
import os
from src.setup import DATA_DIR


df = pd.read_csv(os.path.join(DATA_DIR, "jobs.csv"))

In [13]:
df.created_at = pd.to_datetime(df.created_at)

In [23]:
# KPI

total_found = df.shape[0]
total_found_today = df[
    df.created_at.dt.date == pd.Timestamp.today().date()].shape[0]


pending = df[df.status == "new"].shape[0]
rejected = df[df.status == "removed"].shape[0]
applied = df[df.status == "applied"].shape[0]
seen = df[df.status == "seen"].shape[0]

reviewed = total_found - pending

apply_rate = applied / (reviewed or 1e-6)
reject_rate = rejected / (total_found or 1e-6)



average_relevance_score = df.relevance_score.mean()
average_relevance_score_rejected = df.loc[
    df.status == "removed", "relevance_score"].mean()
average_relevance_score_applied = df.loc[
    df.status == "applied", "relevance_score"].mean()
average_relevance_score_pending = df.loc[
    df.status == "new", "relevance_score"].mean()


# Charts
score_distribution = df.relevance_score
status_breakdown = df.status.value_counts()
job_seniority = df.seniority
employment_type = df.employment_type
easy_apply = df.easy_apply
top_companies = df.company.value_counts()[:10]





In [22]:
df.company.value_counts()[:10]

company
GEICO                   3
CNN                     2
Bristol Myers Squibb    2
Mercor                  2
Lockheed Martin         2
Kforce Inc              2
InterEx Group           2
Glean                   1
Infosys                 1
Quadric                 1
Name: count, dtype: int64

In [17]:
from datetime import datetime


datetime.now().date()

datetime.date(2026, 5, 21)

In [10]:
df.status.value_counts().to_dict()

{'seen': 5, 'new': 4}